# UtilityRAG: Retrieval-Augmented Generation for GIS Conversion Docs

**GeoAI 582 Final Project · Maxx Millstein**

A small LangChain RAG system that answers GIS conversion questions from internal documentation, cites the source file, and says "not in the docs" when it can't find an answer. A 10-question test compares it against the same LLM without retrieval.

The pipeline follows LangChain's [rag-from-scratch](https://github.com/langchain-ai/rag-from-scratch) notebooks (Parts 1–4, plus Multi-Query from Part 5), adapted to Claude + Voyage AI:

| This notebook | rag-from-scratch |
|---|---|
| Part 1: Overview (whole pipeline in one cell) | Part 1 |
| Part 2: Indexing (load, split by heading, embed, store) | Part 2 |
| Part 3: Retrieval | Part 3 |
| Part 4: Generation (cited answers + abstention) | Part 4 |
| Part 5: Baseline (LLM alone) | — |
| Part 6: Evaluation | — |
| Part 7 (optional): Multi-Query retrieval | Part 5 |

**Corpus:** 12 synthetic Markdown docs in `docs/` for a fictional conversion team (GridLine Data Services) and three fictional cooperatives. Key values (field names, codes, tolerances) are deliberately nonstandard so the LLM can't guess them, and the test measures retrieval, not prior knowledge.

The same pipeline is packaged as scripts for the repo: `ingest.py`, `ask.py`, `evaluate.py`, all built on `rag_core.py`.

## Environment
### (1) Packages

In [ ]:
# Run once in the RAG_Project conda env
! pip install langchain langchain-community langchain-anthropic langchain-voyageai langchain-chroma langchain-text-splitters langsmith tiktoken numpy pandas python-dotenv

### (2) API keys

Put these in a `.env` file next to the notebook (it's in `.gitignore`):

```
ANTHROPIC_API_KEY=sk-ant-...
VOYAGE_API_KEY=pa-...
# Optional: LangSmith tracing
LANGCHAIN_API_KEY=lsv2_pt_...
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=utilityrag
```

`load_keys()` also falls back to the `*_api_key.txt` files in this folder.

In [ ]:
import rag_core as rc

rc.load_keys()

### (3) Models

- **LLM:** Claude Haiku 5.5. No `temperature` argument: the Claude API stopped accepting sampling parameters on 2026-10-21.
- **Embeddings:** Voyage `voyage-3.5-lite` (the proposal planned a local Hugging Face model; Voyage is the embedding provider recommended for Claude and needs no GPU).

In [ ]:
llm = rc.get_llm()
embd = rc.get_embeddings()
print(rc.LLM_MODEL, "|", rc.EMBED_MODEL)

## Part 1: Overview

The whole pipeline in one cell, like the rag-from-scratch quickstart. The loader reads the local `docs/` folder instead of a web page. At work this would be a `ConfluenceLoader`; nothing after it changes.

In [35]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

#### INDEXING ####

# Load
docs = [Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
        for p in sorted(Path("docs").glob("*.md"))]

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed (in-memory collection, separate from the main index)
quickstart_store = Chroma.from_documents(documents=splits, embedding=embd,
                                         collection_name="quickstart")
retriever = quickstart_store.as_retriever()

#### RETRIEVAL and GENERATION ####

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
{context}

Question: {question}""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What snapping tolerance should I use for Prairie Wind Rural Electric?")

'Use a **snapping tolerance of 0.25 ft** for Prairie Wind Rural Electric. The source network was digitized from paper, so endpoints are rarely coincident. The QA lead approved this exception, and it applies to all PWRE feature classes. It overrides the Revision 7 default of 0.04 ft, which a cooperative page is allowed to do.'

## Part 2: Indexing

### Load the corpus

In [ ]:
docs = rc.load_docs()
for d in docs:
    print(f"{d.metadata['source']:<36} {len(d.page_content):>5} chars")

### Count tokens

`cl100k_base` is OpenAI's tokenizer, so this is an approximation for Claude/Voyage. It's fine for sizing chunks.

The whole corpus is small enough to fit in one prompt. RAG matters at Confluence scale (hundreds of pages), which the report discusses.

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str = "cl100k_base") -> int:
    return len(tiktoken.get_encoding(encoding_name).encode(string))

total = sum(num_tokens_from_string(d.page_content) for d in docs)
print(f"Corpus: ~{total:,} tokens")

### Split by heading

The quickstart splits every 1,000 characters, which can cut a table in half or separate a value from the heading that says which cooperative it applies to. Conversion docs are organized by heading, so we split on Markdown headings first (`MarkdownHeaderTextSplitter`), then by size only if a section is still too long. Each chunk keeps its file name and heading path for citations.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)

# Example: one doc
for s in header_splitter.split_text(docs[8].page_content):
    print(s.metadata, "|", len(s.page_content), "chars")

In [ ]:
# Full corpus (rc.split_docs = header split + size split + source/section metadata)
chunks = rc.split_docs(docs)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

### Embeddings

As in rag-from-scratch Part 2: embed a question and a passage, and compare with cosine similarity.

In [ ]:
import numpy as np

question = "What snapping tolerance does Prairie Wind use?"
passage = [c for c in chunks if c.metadata["source"] == "08_coop_prairie_wind.md"
           and "Snapping" in c.metadata["section"]][0].page_content
unrelated = [c for c in chunks if c.metadata["source"] == "10_delivery_and_signoff.md"][0].page_content

q_vec = embd.embed_query(question)
p_vec, u_vec = embd.embed_documents([passage, unrelated])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("dimensions:", len(q_vec))
print(f"question vs Prairie Wind snapping section: {cosine_similarity(q_vec, p_vec):.3f}")
print(f"question vs delivery doc:                  {cosine_similarity(q_vec, u_vec):.3f}")

### Store in Chroma

Persisted to `chroma_db/` so `ask.py` and `evaluate.py` can reuse it. Re-running this cell replaces the collection instead of adding duplicate chunks.

In [ ]:
vectorstore = rc.build_vectorstore(chunks, embd)
print(vectorstore._collection.count(), "chunks stored")

## Part 3: Retrieval

Retrieve the top `k = 4` chunks. The scores show whether the cooperative exception outranks the general default, which is the main retrieval challenge in this corpus.

In [ ]:
question = "What snapping tolerance should I use for Prairie Wind Rural Electric?"

for doc, dist in vectorstore.similarity_search_with_score(question, k=rc.TOP_K):
    print(f"{dist:.3f}  {doc.metadata['source']:<32} {doc.metadata['section']}")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": rc.TOP_K})
docs_retrieved = retriever.invoke(question)
len(docs_retrieved)

## Part 4: Generation

The prompt adds three rules to the rag-from-scratch prompt:

1. Cite the source file after each fact.
2. Cooperative pages override general standards.
3. If the context doesn't contain the answer, say exactly **"not in the docs"**.

Each chunk is labeled with its file and section so the model has something to cite.

In [ ]:
print(rc.RAG_TEMPLATE)

In [ ]:
print(rc.format_docs(docs_retrieved[:2]))

In [ ]:
# Chain
chain = rc.rag_prompt | llm | StrOutputParser()
chain.invoke({"context": rc.format_docs(docs_retrieved), "question": question})

### RAG chain with sources

Like rag-from-scratch's `rag_chain_with_source`: retrieve once and return the retrieved documents with the answer, so we can check whether the right file was retrieved.

In [ ]:
rag_chain = rc.build_rag_chain(retriever, llm)

def ask(q):
    out = rag_chain.invoke(q)
    print(out["answer"])
    print("\nRetrieved:", sorted({d.metadata["source"] for d in out["context"]}))
    return out

_ = ask("For Bluestem Valley, how are service points with source status RET mapped?")

In [ ]:
# A question the docs don't cover: should abstain
_ = ask("What is the maximum allowed span length between poles for Bluestem Valley?")

## Part 5: Baseline (LLM alone)

Same model, no retrieval, two versions:

| Baseline | Prompt |
|---|---|
| `baseline` (guarded) | One-line description of GridLine + the same "say *not in the docs* if you don't know" rule as RAG |
| `baseline_open` | The bare question, nothing else. This is how an analyst would use a general chatbot. |

The guarded baseline tests whether the model admits it doesn't know. The open baseline tests the proposal's claim that a general LLM gives a confident, standard answer that is wrong locally.

In [ ]:
baseline_chain = rc.build_baseline_chain(llm)
open_baseline_chain = rc.build_open_baseline_chain(llm)

for q in ["In the GridLine target model, what PHS_CD value is used for a two-phase AC conductor?",
          "For Bluestem Valley, how are service points with source status RET mapped?"]:
    print("Q:", q)
    print("Guarded baseline:", baseline_chain.invoke(q), "\n")
    print("Open baseline:", open_baseline_chain.invoke(q), "\n")

## Part 6: Evaluation

10 questions in `eval/questions.json`:

- 4 **local conventions** (Q01, Q02, Q05, Q06)
- 3 **cooperative exceptions** that override a default (Q03, Q04, Q07)
- 2 **not covered** by the docs (Q08, Q09): the correct answer is "not in the docs"
- 1 **public GIS knowledge** control (Q10)

Each covered question has 1–3 key facts. Scoring is automatic, then reviewed by hand:

| Metric | How it's measured |
|---|---|
| Accuracy | Share of key facts found in the answer (covered questions) |
| Hallucination | Flagged if the system answers an uncovered question, or answers a covered one but misses a key fact. Confirmed by hand in `manual_hallucination`. |
| Abstention | Says "not in the docs" on Q08 and Q09; false abstentions on covered questions are counted separately |
| Retrieval hit | The correct source file is among the top-4 chunks (RAG only). Separates retrieval failures from generation failures. |

In [ ]:
import json
import pandas as pd
from evaluate import run_eval, summarize

questions = json.loads(rc.EVAL_FILE.read_text(encoding="utf-8"))
pd.DataFrame(questions)[["id", "type", "question", "source"]]

In [ ]:
results = run_eval(rag_chain, baseline_chain, questions, open_baseline_chain)
results["manual_hallucination"] = ""  # fill in after reading answers.csv

### Results by question

In [ ]:
pd.set_option("display.max_colwidth", 120)
results.pivot(index="id", columns="system",
              values=["accuracy", "abstained", "hallucination_flag"])

In [ ]:
# RAG retrieval: did the right file come back?
results[results.system == "rag"][["id", "type", "retrieval_hit", "retrieved"]]

### Summary table: baseline vs. RAG

In [ ]:
summary = summarize(results)
summary

### Read the answers side by side

In [ ]:
for qid, g in results.groupby("id"):
    print(f"=== {qid}: {g.question.iloc[0]}")
    for _, r in g.iterrows():
        print(f"[{r.system}] {r.answer}\n")

In [ ]:
rc.RESULTS_DIR.mkdir(exist_ok=True)
results.to_csv(rc.RESULTS_DIR / "answers.csv", index=False)
summary.to_csv(rc.RESULTS_DIR / "summary.csv")
print("Saved to", rc.RESULTS_DIR)

## Part 7 (optional): Multi-Query retrieval

From rag-from-scratch Part 5. The LLM rewrites the question five ways, retrieves for each, and takes the unique union of the results. This can help when an analyst's wording doesn't match the doc's wording (e.g. "tolerance" vs. "snapping"). Compare the retrieval hit rate with Part 6.

In [ ]:
from langchain_core.load import dumps, loads

template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines, with no numbering or other text.
Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)

def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    flattened = [dumps(doc) for sublist in documents for doc in sublist]
    return [loads(doc) for doc in dict.fromkeys(flattened)]

multi_query_retriever = generate_queries | retriever.map() | get_unique_union

docs_mq = multi_query_retriever.invoke("Which tolerance applies to the paper-digitized co-op?")
sorted({d.metadata["source"] for d in docs_mq})

In [ ]:
# Retrieval hit rate: single query vs. multi-query (covered questions only)
rows = []
for item in questions:
    if not item["source"]:
        continue
    single = {d.metadata["source"] for d in retriever.invoke(item["question"])}
    multi = {d.metadata["source"] for d in multi_query_retriever.invoke(item["question"])}
    rows.append({"id": item["id"], "single_hit": item["source"] in single,
                 "multi_hit": item["source"] in multi, "multi_chunks": len(multi)})
mq = pd.DataFrame(rows)
print(mq[["single_hit", "multi_hit"]].mean())
mq

## Notes for the report

- **Why RAG here:** the answers are private, change often (QA Revision 6 → 7), and need a traceable source. Fine-tuning would go stale with every doc update.
- **What RAG is not for:** questions about the data itself (feature counts, current values) or spatial reasoning over geometry. Those need database/PostGIS queries.
- **Scale:** the 12-doc corpus would fit in one prompt. Retrieval matters at Confluence scale, where it also keeps cost and latency down.
- **Limits of the test:** 10 questions show behavior; they are not a statistical benchmark. The automatic hallucination flag is a proxy, confirmed by hand.